# 00 — Environment and GPU

Detect where we are running, confirm PyTorch can see the GPU, and prove it is
actually used by timing a large matrix multiply on both CPU and GPU.

**Run all cells top-to-bottom.** "Restart Kernel and Run All" must pass.

## Before you begin — install PyTorch

Paste the right command into a **terminal** (not a notebook cell) and run it.

```bash
# CUDA 12.1  (NVIDIA GPU on WSL2 — correct for RTX 2050 with recent drivers)
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# CPU-only  (inside a QEMU VM, or if you have no GPU)
pip install torch torchvision torchaudio
```

Do **not** run `pip install` inside this notebook.

**What we do:** Import standard libraries and declare `FORCE_CPU`.
**Why:** A single flag at the top of each notebook controls CPU/GPU; no cell
below needs to inspect hardware again.
**What you should see:** No output — a silent import.

In [1]:
import subprocess, os, pathlib, json, time

# ── configuration ─────────────────────────────────────────────────────────────
FORCE_CPU = False   # set True to ignore the GPU in all cells below
# ──────────────────────────────────────────────────────────────────────────────

**What we do:** Resolve the path to `artifacts/` and create it if needed.
**Why:** Every notebook in this series writes shared files (splits, configs,
checkpoints) there; it must exist before anything tries to write to it.
**What you should see:** The absolute path to `artifacts/`.

In [2]:
# This notebook lives in  neural_nets/notebooks/
# Artifacts live one level up in  neural_nets/artifacts/
NOTEBOOK_DIR  = pathlib.Path().resolve()
ARTIFACTS_DIR = NOTEBOOK_DIR.parent / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"artifacts dir : {ARTIFACTS_DIR}")

artifacts dir : /root/LLM/neural_nets/artifacts


**What we do:** Collect four pieces of evidence — kernel version, `/proc/version`,
`systemd-detect-virt` output, and whether `/dev/dxg` exists.
**Why:** These four signals together distinguish WSL2, a QEMU/KVM guest, and
bare-metal Linux.
**What you should see:** Four labelled lines of raw evidence.

In [3]:
def run(cmd):
    # run a shell command; return (stdout_stripped, returncode)
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=10)
        return r.stdout.strip(), r.returncode
    except Exception as e:
        return str(e), -1

kernel,   _       = run("uname -r")
proc_ver, _       = run("cat /proc/version")
virt,     virt_rc = run("systemd-detect-virt")
has_dxg           = pathlib.Path("/dev/dxg").exists()

print(f"kernel              : {kernel}")
print(f"/proc/version       : {proc_ver[:90]}...")
print(f"systemd-detect-virt : {virt!r}  (rc={virt_rc})")
print(f"/dev/dxg exists     : {has_dxg}")

kernel              : 6.6.114.1-microsoft-standard-WSL2
/proc/version       : Linux version 6.6.114.1-microsoft-standard-WSL2 (root@507f3e43091d) (gcc (GCC) 13.2.0, GNU...
systemd-detect-virt : 'wsl'  (rc=0)
/dev/dxg exists     : True


**What we do:** Run `nvidia-smi` and show its full output.
**Why:** `nvidia-smi` is the ground-truth check — it confirms the NVIDIA driver
is reachable from this environment, not just that a GPU device file exists.
**What you should see:** A table showing GPU name and driver version, or a short
error if the driver is not reachable.

In [4]:
smi_out, smi_rc = run("nvidia-smi")
if smi_rc == 0:
    print(smi_out)
else:
    print(f"nvidia-smi failed (rc={smi_rc})")
    print(smi_out[:300] or "(no output)")

Tue Oct 27 03:16:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.110                Driver Version: 581.86         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2050        On  |   00000000:01:00.0 Off |                  N/A |
| N/A   51C    P3              7W /   35W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**What we do:** Combine the four evidence signals into a single verdict string.
**Why:** One clear label before any PyTorch work starts; also tells you what to
check if the GPU is missing.
**What you should see:** One of:
- *WSL2 with NVIDIA GPU*
- *WSL2 without GPU*
- *QEMU guest, GPU not passed through*
- *unknown*

In [5]:
is_wsl  = "microsoft" in proc_ver.lower()
is_qemu = any(k in virt.lower() for k in ("qemu", "kvm", "bochs"))
gpu_ok  = smi_rc == 0

if is_wsl and gpu_ok:
    verdict = "WSL2 with NVIDIA GPU"
elif is_wsl and not gpu_ok:
    verdict = "WSL2 without GPU"
elif is_qemu:
    verdict = "QEMU guest, GPU not passed through"
else:
    verdict = "unknown"

print(f">>> Environment: {verdict} <<<")
if not gpu_ok:
    print()
    print("Tip: open a Windows terminal and run  nvidia-smi.exe")
    print("If that works, the driver should pass through to WSL2 automatically.")

assert verdict in (
    "WSL2 with NVIDIA GPU",
    "WSL2 without GPU",
    "QEMU guest, GPU not passed through",
    "unknown",
), "FAIL: verdict has an unexpected value"
print()
print("PASS: verdict is one of the four expected strings")

>>> Environment: WSL2 with NVIDIA GPU <<<

PASS: verdict is one of the four expected strings


**What we do:** Import PyTorch and print its version and CUDA availability.
**Why:** A CPU-only PyTorch build reports `cuda.is_available() = False` even
when a GPU is present; the version string confirms which variant is installed.
**What you should see:** Version string, then `True` for CUDA if the driver
is reachable.

In [6]:
import torch

print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version     : {torch.version.cuda}")
    print(f"Device name      : {torch.cuda.get_device_name(0)}")

assert torch.__version__, "FAIL: torch not importable"
print()
print("PASS: torch imported")

PyTorch version  : 2.6.0+cu124
CUDA available   : True
CUDA version     : 12.4
Device name      : NVIDIA GeForce RTX 2050

PASS: torch imported


**What we do:** Print VRAM, compute capability, and check for Intel XPU support.
**Why:** VRAM (4 GB on the RTX 2050) caps the largest batch and model we can load.
Compute capability (8.6 for RTX 30xx) determines which CUDA ops are available.
XPU is reported for completeness but not used — we don't install Intel's GPU stack.
**What you should see:** Two numbers if a CUDA GPU is present, otherwise a note.

In [7]:
if torch.cuda.is_available():
    props   = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1024 ** 3
    cc      = torch.cuda.get_device_capability(0)
    print(f"VRAM               : {vram_gb:.1f} GB")
    print(f"Compute capability : {cc[0]}.{cc[1]}")
else:
    print("No CUDA GPU found. All models will run on CPU.")

if hasattr(torch, "xpu"):
    print(f"torch.xpu available: {torch.xpu.is_available()}")
else:
    print("torch.xpu          : not in this build (Intel GPU stack not installed)")

VRAM               : 4.0 GB
Compute capability : 8.6
torch.xpu available: False


**What we do:** Set the `device` variable used by all remaining cells and by
later notebooks that read `env.json`.
**Why:** One variable, set once; `FORCE_CPU = True` at the top overrides even
a working GPU without touching any other cell.
**What you should see:** A line confirming which device was chosen.

In [8]:
if FORCE_CPU:
    device = torch.device("cpu")
    print("FORCE_CPU=True  ->  running on CPU")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("No GPU found  ->  running on CPU")

print(f"device = {device}")
assert str(device) in ("cpu", "cuda"), "FAIL: unexpected device string"
print()
print("PASS: device set")

Using GPU: NVIDIA GeForce RTX 2050
device = cuda

PASS: device set


**What we do:** Multiply two 4096 x 4096 random matrices on the CPU, then on
the GPU (after a warmup pass), and compare wall-clock times.
**Why:** Detection is not the same as use — this proves the GPU is actually
computing. The warmup pass is needed because the first GPU kernel call includes
JIT compilation overhead that would unfairly inflate the measured time.
**What you should see:** CPU and GPU times in milliseconds, and a speedup ratio.
A 4096 x 4096 matmul is large enough that the RTX 2050 should be noticeably
faster than the CPU.

In [9]:
torch.manual_seed(42)
N = 4096

a_cpu = torch.randn(N, N)
b_cpu = torch.randn(N, N)

# ── CPU ───────────────────────────────────────────────────────────────────────
t0 = time.perf_counter()
_ = a_cpu @ b_cpu
cpu_ms = (time.perf_counter() - t0) * 1000
print(f"CPU  {N}x{N} matmul : {cpu_ms:8.1f} ms")

# ── GPU ───────────────────────────────────────────────────────────────────────
if device.type == "cuda":
    a_gpu = a_cpu.to(device)
    b_gpu = b_cpu.to(device)

    _ = a_gpu @ b_gpu            # warmup: first call includes JIT/kernel launch overhead
    torch.cuda.synchronize()

    t0 = time.perf_counter()
    _ = a_gpu @ b_gpu
    torch.cuda.synchronize()
    gpu_ms = (time.perf_counter() - t0) * 1000

    print(f"GPU  {N}x{N} matmul : {gpu_ms:8.1f} ms")
    print(f"Speedup            : {cpu_ms / gpu_ms:.1f}x")

    assert gpu_ms < cpu_ms, (
        f"FAIL: GPU ({gpu_ms:.1f} ms) was not faster than CPU ({cpu_ms:.1f} ms)"
    )
    print()
    print("PASS: GPU is faster than CPU for 4096x4096 matmul")
else:
    print("No GPU available  -  skipping GPU benchmark.")
    print()
    print("PASS: CPU-only path ran without error")

CPU  4096x4096 matmul :    879.4 ms


GPU  4096x4096 matmul :     46.3 ms
Speedup            : 19.0x

PASS: GPU is faster than CPU for 4096x4096 matmul


**What we do:** Write all findings to `artifacts/env.json`, then read the file
back and verify the round-trip.
**Why:** Notebooks 01–04 read `env.json` to know which `device` to use, without
repeating environment detection.
**What you should see:** The JSON printed to the screen, then PASS.

In [10]:
env = {
    "verdict"         : verdict,
    "device"          : str(device),
    "kernel"          : kernel,
    "is_wsl"          : is_wsl,
    "pytorch_version" : torch.__version__,
    "cuda_available"  : torch.cuda.is_available(),
}
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    cc    = torch.cuda.get_device_capability(0)
    env["gpu_name"]           = torch.cuda.get_device_name(0)
    env["vram_gb"]            = round(props.total_memory / 1024 ** 3, 1)
    env["compute_capability"] = f"{cc[0]}.{cc[1]}"

env_path = ARTIFACTS_DIR / "env.json"
with open(env_path, "w") as f:
    json.dump(env, f, indent=2)

print(json.dumps(env, indent=2))

# verify round-trip
with open(env_path) as f:
    loaded = json.load(f)
assert loaded["device"] == str(device), "FAIL: env.json did not round-trip correctly"
print()
print(f"PASS: env.json written and verified -> {env_path}")

{
  "verdict": "WSL2 with NVIDIA GPU",
  "device": "cuda",
  "kernel": "6.6.114.1-microsoft-standard-WSL2",
  "is_wsl": true,
  "pytorch_version": "2.6.0+cu124",
  "cuda_available": true,
  "gpu_name": "NVIDIA GeForce RTX 2050",
  "vram_gb": 4.0,
  "compute_capability": "8.6"
}

PASS: env.json written and verified -> /root/LLM/neural_nets/artifacts/env.json
